In [7]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Dense,
    Flatten,
    GlobalAveragePooling2D,
    Input
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [8]:
IMG_SIZE = 128
BATCH_SIZE = 16
NUM_CLASSES = 7

In [9]:
#Load Dataset

train_df = pd.read_csv("Datasets/train.csv")
valid_df = pd.read_csv("Datasets/validation.csv")
test_df = pd.read_csv("Datasets/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)
metadata = pd.read_csv("Datasets/HAM10000_metadata.csv")

metadata.head()

(7010, 11)
(1502, 11)
(1503, 11)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [10]:
image_dir1 = "Datasets/HAM10000_images_part_1"
image_dir2 = "Datasets/HAM10000_images_part_2"

image_path = {}

for folder in [image_dir1, image_dir2]:

    for file in os.listdir(folder):

        image_id = file.split(".")[0]

        image_path[image_id] = os.path.join(folder, file)

metadata["path"] = metadata["image_id"].map(image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2/ISIC_0031633.jpg


In [13]:
encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0027419.jpg,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025030.jpg,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0026769.jpg,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025661.jpg,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2/ISIC_0031633.jpg,2


In [14]:
train_df, temp_df = train_test_split(

    metadata,

    test_size=0.30,

    stratify=metadata["label"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 9)
(1502, 9)
(1503, 9)


In [15]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [16]:
train_df["path"] = train_df["path"].astype(str)
val_df["path"] = val_df["path"].astype(str)
test_df["path"] = test_df["path"].astype(str)

In [17]:
classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [18]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.2,

    horizontal_flip=True,

    vertical_flip=True
)

test_datagen = ImageDataGenerator(

    rescale=1./255
)

In [19]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [20]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    horizontal_flip=True,

    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]

)

test_datagen = ImageDataGenerator(

    rescale=1./255

)

In [21]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True

)

Found 7010 validated image filenames belonging to 7 classes.


In [62]:
from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# NOW create the generators
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(128,128),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(128,128),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(128,128),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


# EARLY STOP

In [63]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze all pretrained layers
base_model.trainable = False

resnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),   # Reduced from 256 to 128
    Dropout(0.3),                    # Reduced from 0.5 to 0.3
    Dense(NUM_CLASSES, activation="softmax")
])

resnet.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [64]:
#Compile
resnet.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [65]:
early_stop = EarlyStopping(

    monitor="val_accuracy",

    mode="max",

    patience=3,

    restore_best_weights=True,

    verbose=1

)

In [66]:
#Train
history_ES= resnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,
    callbacks=[early_stop],

    class_weight=class_weights)



Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 75s 166ms/step - accuracy: 0.3565 - loss: 1.8555 - val_accuracy: 0.3975 - val_loss: 1.5440
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 76s 174ms/step - accuracy: 0.4429 - loss: 1.5019 - val_accuracy: 0.5260 - val_loss: 1.3191
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 167ms/step - accuracy: 0.4922 - loss: 1.3711 - val_accuracy: 0.5293 - val_loss: 1.2393
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 166ms/step - accuracy: 0.4994 - loss: 1.3037 - val_accuracy: 0.5413 - val_loss: 1.2159
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 72s 163ms/step - accuracy: 0.5230 - loss: 1.2516 - val_accuracy: 0.5619 - val_loss: 1.1487
Restoring model weights from the end of the best epoch: 5.


In [67]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = resnet.evaluate(train_generator, verbose=0)

val_loss, val_acc = resnet.evaluate(val_generator, verbose=0)

test_loss, test_acc = resnet.evaluate(test_generator, verbose=0)

pred = resnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [68]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.6017118692398071 0.5619174242019653 0.5695276260375977 0.7407484559451191 0.5695276114437791 0.6180862239891751


In [69]:
resnet.save("models2/resnet_EARLY.keras")

# lr

In [71]:
IMG_SIZE=128
#ResNet50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE,IMG_SIZE,3)

)

base_model.trainable = False

resnet = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),
    Dropout(0.5),

    Dense(NUM_CLASSES,activation="softmax")

])

resnet.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,055 (91.99 MB)

 Trainable params: 526,343 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [72]:
#Compile
resnet.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [73]:
reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-6,

    verbose=1

)

In [74]:
history_lr = resnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[reduce_lr]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 76s 169ms/step - accuracy: 0.3756 - loss: 1.8253 - val_accuracy: 0.4747 - val_loss: 1.3191 - learning_rate: 5.0000e-04
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 166ms/step - accuracy: 0.4558 - loss: 1.4291 - val_accuracy: 0.5859 - val_loss: 1.1461 - learning_rate: 5.0000e-04
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 75s 171ms/step - accuracy: 0.4800 - loss: 1.3382 - val_accuracy: 0.5546 - val_loss: 1.1439 - learning_rate: 5.0000e-04
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 72s 163ms/step - accuracy: 0.5136 - loss: 1.2633 - val_accuracy: 0.5253 - val_loss: 1.1852 - learning_rate: 5.0000e-04
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 71s 161ms/step - accuracy: 0.5160 - loss: 1.2519 - val_accuracy: 0.6092 - val_loss: 1.0123 - learning_rate: 5.0000e-04


In [75]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = resnet.evaluate(train_generator, verbose=0)

val_loss, val_acc = resnet.evaluate(val_generator, verbose=0)

test_loss, test_acc = resnet.evaluate(test_generator, verbose=0)

pred = resnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [76]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.6276747584342957 0.6091877222061157 0.6114437580108643 0.7425476027833086 0.6114437791084497 0.6528130592674855


In [77]:
resnet.save("models2/resnetmodel_Lr.keras")

# SGD

In [78]:
#ResNet50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE,IMG_SIZE,3)

)

base_model.trainable = False

resnet = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),
    Dropout(0.5),

    Dense(NUM_CLASSES,activation="softmax")

])

resnet.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,055 (91.99 MB)

 Trainable params: 526,343 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [79]:
from tensorflow.keras.optimizers import SGD

resnet.compile(

    optimizer=SGD(

        learning_rate=0.0001,

        momentum=0.9,

        nesterov=True

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [80]:
history_lr = resnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[reduce_lr]

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 163ms/step - accuracy: 0.2785 - loss: 2.0164 - val_accuracy: 0.4001 - val_loss: 1.5530 - learning_rate: 1.0000e-04
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 74s 168ms/step - accuracy: 0.3849 - loss: 1.6136 - val_accuracy: 0.4334 - val_loss: 1.5018 - learning_rate: 1.0000e-04
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 71s 161ms/step - accuracy: 0.4205 - loss: 1.5334 - val_accuracy: 0.4234 - val_loss: 1.4493 - learning_rate: 1.0000e-04
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 71s 161ms/step - accuracy: 0.4357 - loss: 1.4531 - val_accuracy: 0.4268 - val_loss: 1.4400 - learning_rate: 1.0000e-04
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 71s 161ms/step - accuracy: 0.4327 - loss: 1.4333 - val_accuracy: 0.5113 - val_loss: 1.2741 - learning_rate: 1.0000e-04


In [81]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = resnet.evaluate(train_generator, verbose=0)

val_loss, val_acc = resnet.evaluate(val_generator, verbose=0)

test_loss, test_acc = resnet.evaluate(test_generator, verbose=0)

pred = resnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [82]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.5291013121604919 0.5113182663917542 0.514970064163208 0.7417848408105361 0.5149700598802395 0.5830718125640545


In [83]:
resnet.save("models2/resnetmodel_rms.keras")

# RMS

In [84]:
#ResNet50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE,IMG_SIZE,3)

)

base_model.trainable = False

resnet = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),
    Dropout(0.5),

    Dense(NUM_CLASSES,activation="softmax")

])

resnet.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_8      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,055 (91.99 MB)

 Trainable params: 526,343 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [85]:
from tensorflow.keras.optimizers import RMSprop

resnet.compile(

    optimizer=RMSprop(

        learning_rate=0.001

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [86]:
history_dense = resnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 77s 171ms/step - accuracy: 0.4679 - loss: 2.2171 - val_accuracy: 0.5399 - val_loss: 1.2151
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 167ms/step - accuracy: 0.5275 - loss: 1.9275 - val_accuracy: 0.6258 - val_loss: 0.9960
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 167ms/step - accuracy: 0.5459 - loss: 1.8108 - val_accuracy: 0.4740 - val_loss: 1.3913
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 76s 172ms/step - accuracy: 0.5491 - loss: 1.8390 - val_accuracy: 0.6025 - val_loss: 1.0832
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 75s 170ms/step - accuracy: 0.5511 - loss: 1.7631 - val_accuracy: 0.5632 - val_loss: 1.0903


In [87]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = resnet.evaluate(train_generator, verbose=0)

val_loss, val_acc = resnet.evaluate(val_generator, verbose=0)

test_loss, test_acc = resnet.evaluate(test_generator, verbose=0)

pred = resnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [88]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.5932952761650085 0.5632489919662476 0.5608782172203064 0.7460455927195594 0.5608782435129741 0.6144496205411637


In [89]:
resnet.save("models2/resnet_model_Realrms.keras")

# batch

In [90]:
train_generator_32 = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=True

)

val_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

test_generator_32 = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=32,

    class_mode="categorical",

    shuffle=False

)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [91]:
#ResNet50
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential

base_model = ResNet50(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE,IMG_SIZE,3)

)

base_model.trainable = False

resnet = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256,activation="relu"),
    Dropout(0.5),

    Dense(NUM_CLASSES,activation="softmax")

])

resnet.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_9      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,055 (91.99 MB)

 Trainable params: 526,343 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [92]:
from tensorflow.keras.optimizers import RMSprop

resnet.compile(

    optimizer=RMSprop(

        learning_rate=0.001

    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [93]:
history_dense = resnet.fit(

    train_generator_32,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    verbose=1

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 325ms/step - accuracy: 0.4466 - loss: 2.0321 - val_accuracy: 0.6312 - val_loss: 1.0475
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 71s 321ms/step - accuracy: 0.5021 - loss: 1.7032 - val_accuracy: 0.5652 - val_loss: 1.1888
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 70s 320ms/step - accuracy: 0.5325 - loss: 1.5822 - val_accuracy: 0.5486 - val_loss: 1.1267
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 70s 319ms/step - accuracy: 0.5438 - loss: 1.5557 - val_accuracy: 0.6045 - val_loss: 1.0296
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 71s 322ms/step - accuracy: 0.5511 - loss: 1.5088 - val_accuracy: 0.5792 - val_loss: 1.1236


In [94]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = resnet.evaluate(train_generator, verbose=0)

val_loss, val_acc = resnet.evaluate(val_generator, verbose=0)

test_loss, test_acc = resnet.evaluate(test_generator, verbose=0)

pred = resnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [95]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.6008559465408325 0.5792276859283447 0.5788423418998718 0.7490558842990475 0.5788423153692615 0.6315806083437108


In [96]:
resnet.save("models2/resnet_mode_batch.keras")

# Finetuning

In [97]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

IMG_SIZE = (224, 224)
NUM_CLASSES = 7

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(128,128, 3)
)
base_model.trainable = False
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

history = model.fit(

    train_generator,

    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 76s 168ms/step - accuracy: 0.6884 - loss: 0.9415 - val_accuracy: 0.7310 - val_loss: 0.7593 - learning_rate: 0.0010
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 167ms/step - accuracy: 0.7130 - loss: 0.7960 - val_accuracy: 0.7437 - val_loss: 0.7122 - learning_rate: 0.0010
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 167ms/step - accuracy: 0.7237 - loss: 0.7642 - val_accuracy: 0.7490 - val_loss: 0.7071 - learning_rate: 0.0010
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 74s 167ms/step - accuracy: 0.7335 - loss: 0.7472 - val_accuracy: 0.7597 - val_loss: 0.6986 - learning_rate: 0.0010
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 166ms/step - accuracy: 0.7378 - loss: 0.7233 - val_accuracy: 0.7530 - val_loss: 0.7288 - learning_rate: 0.0010
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 73s 166ms/step - accuracy: 0.7411 - loss: 0.7140 - val_accuracy: 0.7623 - val_loss: 0.6685 - learning_rate: 0.0010
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 75s 172ms/step - accuracy: 0.7

In [98]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = model.evaluate(train_generator, verbose=0)

val_loss, val_acc = model.evaluate(val_generator, verbose=0)

test_loss, test_acc = model.evaluate(test_generator, verbose=0)

pred = model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [99]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.7748929858207703 0.7663115859031677 0.7611443996429443 0.7337351610591444 0.761144377910845 0.7369274589962682


In [100]:
model.save("models2/resnet50_finetuned.keras")

# Hypertuning

In [101]:
pip install keras-tuner

Note: you may need to restart the kernel to use updated packages.


In [102]:
import tensorflow as tf
import keras_tuner as kt

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

In [103]:
IMG_SIZE = (128,128)
NUM_CLASSES = 7

def build_resnet_model(hp):

    base_model = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(128, 128, 3)
    )

    # Freeze pretrained layers
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    x = Dense(
        units=hp.Choice("dense_units", [128, 256, 512]),
        activation="relu"
    )(x)

    x = Dropout(
        hp.Float("dropout", 0.2, 0.5, step=0.1)
    )(x)

    outputs = Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(base_model.input, outputs)

    lr = hp.Choice(
        "learning_rate",
        [1e-2, 1e-3, 1e-4]
    )

    optimizer_name = hp.Choice(
        "optimizer",
        ["adam", "rmsprop"]
    )

    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    else:
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [104]:
tuner = kt.RandomSearch(
    build_resnet_model,
    objective="val_accuracy",
    max_trials=5,          # Keep small for CPU
    executions_per_trial=1,
    directory="resnet_tuning",
    project_name="skin_disease"
)

Reloading Tuner from resnet_tuning/skin_disease/tuner0.json


In [105]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [106]:
tuner.search(
    train_generator,

    validation_data=val_generator,
    epochs=5,
    callbacks=[early_stop],
    class_weights= class_weights
)

In [107]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print("Dense Units :", best_hp.get("dense_units"))
print("Dropout     :", best_hp.get("dropout"))
print("Learning Rate:", best_hp.get("learning_rate"))
print("Optimizer   :", best_hp.get("optimizer"))

Dense Units : 256
Dropout     : 0.30000000000000004
Learning Rate: 0.01
Optimizer   : rmsprop


In [108]:
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(
    train_generator,

    validation_data=val_generator,
    epochs=5,
    callbacks=[early_stop]
)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 76s 170ms/step - accuracy: 0.6576 - loss: 1.5139 - val_accuracy: 0.6778 - val_loss: 0.9714
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 82s 187ms/step - accuracy: 0.6745 - loss: 1.1006 - val_accuracy: 0.6778 - val_loss: 1.1367
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 75s 171ms/step - accuracy: 0.6816 - loss: 1.0505 - val_accuracy: 0.6984 - val_loss: 0.9637
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 72s 165ms/step - accuracy: 0.6839 - loss: 1.0781 - val_accuracy: 0.6824 - val_loss: 1.0078
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 83s 189ms/step - accuracy: 0.6825 - loss: 1.0922 - val_accuracy: 0.6917 - val_loss: 1.0929


In [109]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = model.evaluate(train_generator, verbose=0)

val_loss, val_acc = model.evaluate(val_generator, verbose=0)

test_loss, test_acc = model.evaluate(test_generator, verbose=0)

pred = model.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [110]:
print( train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1)

0.778744637966156 0.7663115859031677 0.7611443996429443 0.7337351610591444 0.761144377910845 0.7369274589962682


In [114]:
model.save("models2/resnet50_finetuned.keras")

In [112]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Model": [
        "Early Stopping",
        "Learning Rate Scheduler",
        "SGD",
        "RMSprop",
        "Batch-wise Comparison",
        "Fine-Tuning",
        "Hyperparameter Tuning"
    ],
    "Train Accuracy": [
        0.6017118692398071,
        0.6276747584342957,
        0.5291013121604919,
        0.5932952761650085,
        0.6008559465408325,
        0.7748929858207703,
        0.778744637966156
    ],
    "Validation Accuracy": [
        0.5619174242019653,
        0.6091877222061157,
        0.5113182663917542,
        0.5632489919662476,
        0.5792276859283447,
        0.7663115859031677,
        0.7663115859031677
    ],
    "Test Accuracy": [
        0.5695276260375977,
        0.6114437580108643,
        0.514970064163208,
        0.5608782172203064,
        0.5788423418998718,
        0.7611443996429443,
        0.7611443996429443
    ],
    "Precision": [
        0.7407484559451191,
        0.7425476027833086,
        0.7417848408105361,
        0.7460455927195594,
        0.7490558842990475,
        0.7337351610591444,
        0.7337351610591444
    ],
    "Recall": [
        0.5695276114437791,
        0.6114437791084497,
        0.5149700598802395,
        0.5608782435129741,
        0.5788423153692615,
        0.761144377910845,
        0.761144377910845
    ],
    "F1-Score": [
        0.6180862239891751,
        0.6528130592674855,
        0.5830718125640545,
        0.6144496205411637,
        0.6315806083437108,
        0.7369274589962682,
        0.7369274589962682
    ]
})

# Sort by Test Accuracy (Highest to Lowest)
comparison_df = (
    comparison_df.sort_values("Test Accuracy", ascending=False)
                 .reset_index(drop=True)
                 .round(4)
)

comparison_df

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Precision,Recall,F1-Score
0,Fine-Tuning,0.7749,0.7663,0.7611,0.7337,0.7611,0.7369
1,Hyperparameter Tuning,0.7787,0.7663,0.7611,0.7337,0.7611,0.7369
2,Learning Rate Scheduler,0.6277,0.6092,0.6114,0.7425,0.6114,0.6528
3,Batch-wise Comparison,0.6009,0.5792,0.5788,0.7491,0.5788,0.6316
4,Early Stopping,0.6017,0.5619,0.5695,0.7407,0.5695,0.6181
5,RMSprop,0.5933,0.5632,0.5609,0.7460,0.5609,0.6144
6,SGD,0.5291,0.5113,0.5150,0.7418,0.5150,0.5831
